# HireFlow 系统评估报告

**报告日期:** 自动生成  
**报告时间:** 自动生成  
**项目:** HireFlow - 基于 LangGraph 的多 Agent 招聘筛选系统  
**Phase:** 1.1 核心 Pipeline

In [ ]:
# ================================================================
# Cell 1: 报告元信息 (日期 + 具体时间)
# ================================================================
from datetime import datetime
import pytz

# 获取当前时间 (悉尼时区)
sydney_tz = pytz.timezone("Australia/Sydney")
now = datetime.now(sydney_tz)

# 格式化为要求的时间格式
report_date = now.strftime("%Y年%m月%d日")
report_time = now.strftime("%I:%M %p")  # 例如: "02:30 PM"

print("=" * 60)
print("  HireFlow 系统评估报告")
print("=" * 60)
print(f"  报告日期: {report_date}")
print(f"  报告时间: {report_time}")
print(f"  时区:     Australia/Sydney (悉尼)")
print("=" * 60)

---
## 一、系统架构概览

In [ ]:
# ================================================================
# Cell 2: 系统架构图 (ASCII 可视化)
# ================================================================

architecture = """
┌─────────────────────────────────────────────────────────────────────┐
│                        HireFlow 系统架构                              │
│                     Phase 1.1 — 核心 Pipeline                         │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐     │
│  │ 用户输入  │───▶│ JD Agent │───▶│ Resume   │───▶│  Match   │     │
│  │ (CLI/API) │    │ 解析JD   │    │ Agent    │    │  Agent   │     │
│  └──────────┘    └──────────┘    │ 解析简历  │    │ 匹配评分  │     │
│                                  └──────────┘    └─────┬────┘     │
│                                                        │          │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐         │          │
│  │ 结果展示  │◀───│  Ranking │◀───│   RAG    │◀────────┘          │
│  │ (排序表)  │    │  Agent   │    │ 证据检索  │                     │
│  └──────────┘    └──────────┘    └──────────┘                    │
│                                                                     │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  数据层                                                      │   │
│  │  ┌────────────┐  ┌────────────┐  ┌────────────────────┐    │   │
│  │  │ PostgreSQL │  │   Qdrant   │  │  文件系统 (.md)    │    │   │
│  │  │ 关系数据    │  │  向量检索   │  │  测试数据           │    │   │
│  │  └────────────┘  └────────────┘  └────────────────────┘    │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  LLM 后端 (双模式)                                            │   │
│  │  ┌──────────────────┐    ┌──────────────────────────┐      │   │
│  │  │ LM Studio (本地)  │    │ DeepSeek API (云端)       │      │   │
│  │  │ hermes-3-8b       │    │ deepseek-v4-pro           │      │   │
│  │  │ localhost:1234    │    │ api.deepseek.com          │      │   │
│  │  └──────────────────┘    └──────────────────────────┘      │   │
│  └─────────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────────┘
"""
print(architecture)

---
## 二、系统状态检查

In [ ]:
# ================================================================
# Cell 3: 配置检查
# ================================================================
print("【配置检查】")
print(f"  报告时间: {report_time}")
print()

from app.utils.config import settings

print(f"  LLM 模式:     {settings.llm.mode}")
print(f"  本地模型:      {settings.llm.local_model}")
print(f"  云端模型:      {settings.llm.cloud_model}")
print(f"  本地Base URL:  {settings.llm.local_base_url}")
print(f"  云端Base URL:  {settings.llm.cloud_base_url}")
print(f"  LLM max_tokens:{settings.llm.max_tokens}")
print(f"  LLM temperature:{settings.llm.temperature}")
print()
print(f"  Embedding模式: {settings.embedding.mode}")
print(f"  本地Embedding:  {settings.embedding.local_model}")
print(f"  Embedding维度: {settings.embedding.dimension}")
print()
print(f"  数据库URL:     {settings.database.url}")
print(f"  Qdrant URL:    {settings.qdrant.url}")
print(f"  Chunk大小:     {settings.document.chunk_size}")
print(f"  Chunk重叠:     {settings.document.chunk_overlap}")

# 检查 API Key 是否配置
if settings.llm.mode == "cloud" and not settings.llm.cloud_api_key:
    print()
    print("  ⚠️  警告: 云端模式已启用但未配置 API Key!")
    print("  ⚠️  请在 .env 文件中设置 LLM_CLOUD_API_KEY")

print()
print("  配置检查完成 ✅")

In [ ]:
# ================================================================
# Cell 4: LLM 连接测试
# ================================================================
print("【LLM 连接测试】")
print(f"  测试时间: {report_time}")
print()

from openai import OpenAI

def test_llm_connection(name: str, base_url: str, api_key: str, model: str):
    """测试 LLM 连接是否正常。"""
    try:
        client = OpenAI(base_url=base_url, api_key=api_key)
        # 列出可用模型
        models_resp = client.models.list()
        models = [m.id for m in models_resp.data]
        
        # 检查目标模型是否在列表中
        if model in models:
            print(f"  ✅ {name}: 模型 '{model}' 已就绪")
        else:
            matched = [m for m in models if model.lower() in m.lower()]
            if matched:
                print(f"  ⚠️  {name}: 模型名不完全匹配，可用: {matched}")
            else:
                print(f"  ⚠️  {name}: 模型 '{model}' 未找到，可用模型: {models[:3]}...")
        
        # 快速对话测试
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=10,
        )
        print(f"      对话测试: OK (tokens={resp.usage.total_tokens})")
        
        return True
    except Exception as e:
        print(f"  ❌ {name}: 连接失败 - {str(e)[:100]}")
        return False

# 测试本地 LLM
local_ok = test_llm_connection(
    "本地 LM Studio",
    settings.llm.local_base_url,
    settings.llm.local_api_key,
    settings.llm.local_model,
)

# 测试本地 Embedding
print()
emb_ok = False
try:
    client = OpenAI(base_url=settings.embedding.local_base_url, api_key=settings.embedding.local_api_key)
    resp = client.embeddings.create(model=settings.embedding.local_model, input="test")
    dim = len(resp.data[0].embedding)
    print(f"  ✅ 本地 Embedding: '{settings.embedding.local_model}' 就绪 (维度={dim})")
    emb_ok = True
except Exception as e:
    print(f"  ❌ 本地 Embedding: 连接失败 - {str(e)[:100]}")

print()
if local_ok and emb_ok:
    print("  LLM 连接测试全部通过 ✅")
else:
    print(f"  LLM 连接测试有问题 (LLM={'✅' if local_ok else '❌'}, Embedding={'✅' if emb_ok else '❌'})")

In [ ]:
# ================================================================
# Cell 5: 数据库连接测试
# ================================================================
print("【数据库连接测试】")
print(f"  测试时间: {report_time}")
print()

from app.database.session import init_db, SessionLocal, engine

# 测试 PostgreSQL
try:
    # 尝试连接
    connection = engine.connect()
    print(f"  ✅ PostgreSQL: 连接成功 ({settings.database.url.split('@')[1] if '@' in settings.database.url else settings.database.url})")
    connection.close()
    
    # 初始化表
    init_db()
    print(f"  ✅ 数据库表: 已初始化")
    
    # 查询表列表
    from sqlalchemy import inspect
    inspector = inspect(engine)
    tables = inspector.get_table_names()
    print(f"  ✅ 现有表: {', '.join(tables)}")
    
    db_ok = True
except Exception as e:
    print(f"  ❌ PostgreSQL: 连接失败 - {str(e)[:150]}")
    print(f"  💡 提示: 请运行 'docker compose up -d postgres' 启动数据库")
    db_ok = False

# 测试 Qdrant
print()
try:
    from qdrant_client import QdrantClient
    client = QdrantClient(url=settings.qdrant.url)
    collections = client.get_collections()
    names = [c.name for c in collections.collections]
    print(f"  ✅ Qdrant: 连接成功 ({settings.qdrant.url})")
    print(f"  ✅ 现有集合: {names if names else '(空)'}")
    qdrant_ok = True
except Exception as e:
    print(f"  ❌ Qdrant: 连接失败 - {str(e)[:150]}")
    print(f"  💡 提示: 请运行 'docker compose up -d qdrant' 启动向量数据库")
    qdrant_ok = False

print()
if db_ok and qdrant_ok:
    print("  数据库连接测试全部通过 ✅")
else:
    print(f"  数据库连接有问题 (PostgreSQL={'✅' if db_ok else '❌'}, Qdrant={'✅' if qdrant_ok else '❌'})")

---
## 三、Pipeline 运行测试

In [ ]:
# ================================================================
# Cell 6: 单步骤测试 (JD解析)
# ================================================================
print("【Pipeline 单步测试】")
print(f"  测试时间: {report_time}")
print()

import asyncio
import time

test_jd = """
岗位名称: 初级 Python 后端开发工程师
必备技能: Python, FastAPI, PostgreSQL, Docker
加分技能: LangChain, RAG
学历要求: 计算机相关专业本科
"""

test_resume = """
姓名: 测试候选人
教育: 2020-2024 清华大学 计算机科学 学士
技能: Python, FastAPI, Docker, PostgreSQL
项目: API网关 - 使用FastAPI开发的高性能API网关
"""

async def run_step_tests():
    from app.agents.jd_agent import analyze_jd
    from app.agents.resume_agent import parse_resume
    
    results = {}
    
    # 测试 JD 解析
    print("  1. JD 解析测试...")
    start = time.time()
    try:
        jd_profile = await analyze_jd(test_jd)
        elapsed = time.time() - start
        print(f"     ✅ 成功 (耗时: {elapsed:.1f}秒)")
        print(f"     岗位: {jd_profile.get('job_title', '?')}")
        print(f"     必备技能: {jd_profile.get('required_skills', [])}")
        results['jd'] = {'status': 'ok', 'time': elapsed}
    except Exception as e:
        elapsed = time.time() - start
        print(f"     ❌ 失败 (耗时: {elapsed:.1f}秒): {str(e)[:100]}")
        results['jd'] = {'status': 'fail', 'time': elapsed, 'error': str(e)[:100]}
    
    # 测试 Resume 解析
    print()
    print("  2. Resume 解析测试...")
    start = time.time()
    try:
        profile = await parse_resume(test_resume, "TEST001")
        elapsed = time.time() - start
        print(f"     ✅ 成功 (耗时: {elapsed:.1f}秒)")
        print(f"     姓名: {profile.get('name', '?')}")
        print(f"     技能数: {len(profile.get('skills', []))}")
        results['resume'] = {'status': 'ok', 'time': elapsed}
    except Exception as e:
        elapsed = time.time() - start
        print(f"     ❌ 失败 (耗时: {elapsed:.1f}秒): {str(e)[:100]}")
        results['resume'] = {'status': 'fail', 'time': elapsed, 'error': str(e)[:100]}
    
    return results

step_results = asyncio.run(run_step_tests())

print()
print(f"  单步测试完成 (JD={'✅' if step_results.get('jd',{}).get('status')=='ok' else '❌'}, "
      f"Resume={'✅' if step_results.get('resume',{}).get('status')=='ok' else '❌'})")

In [ ]:
# ================================================================
# Cell 7: 完整 Pipeline 运行测试
# ================================================================
print("【完整 Pipeline 测试】")
print(f"  测试时间: {report_time}")
print()

async def run_full_pipeline_test():
    from app.agents.jd_agent import analyze_jd
    from app.agents.resume_agent import batch_parse_resumes
    from app.agents.match_agent import batch_match_candidates
    from app.agents.ranking_agent import rank_candidates
    
    total_start = time.time()
    timeline = []  # 记录每个步骤的时间
    
    # 完整 JD
    full_jd = """
岗位名称: Python 后端开发工程师
必备技能: Python, FastAPI, PostgreSQL, Docker, Git
加分技能: LangChain, RAG, Redis, Kubernetes
岗位职责: 开发后端API, 数据库设计, 编写单元测试
学历要求: 计算机相关专业本科及以上
经验要求: 0-3年
"""
    
    # 3份测试简历
    test_resumes = {
        "E001": "姓名: 张工\n技能: Python, FastAPI, PostgreSQL, Docker, Git, Redis\n项目: 电商API - FastAPI+PostgreSQL+Docker\n教育: 2020-2024 北大 CS学士\n经历: 2023某公司Python实习生",
        "E002": "姓名: 李工\n技能: Python, Django, MySQL, Docker, Git\n项目: 博客系统 - Django+MySQL\n教育: 2019-2023 浙大 SE学士\n经历: 2022某公司Django实习生",
        "E003": "姓名: 王工\n技能: Python, FastAPI, PostgreSQL, LangChain, RAG, Docker, Git, Redis\n项目: RAG问答系统 - FastAPI+LangChain+Qdrant\n教育: 2021-2023 清华 AI硕士\n经历: 2023某AI公司后端实习生",
    }
    
    # Step 1: JD 解析
    print("  [1/4] JD解析...")
    start = time.time()
    jd_profile = await analyze_jd(full_jd)
    t1 = time.time() - start
    timeline.append(("JD解析", t1, "ok"))
    print(f"        耗时: {t1:.1f}秒")
    
    # Step 2: 简历解析
    print("  [2/4] 简历解析...")
    start = time.time()
    profiles = await batch_parse_resumes(test_resumes)
    t2 = time.time() - start
    timeline.append(("简历解析", t2, "ok"))
    print(f"        耗时: {t2:.1f}秒 ({len(profiles)}份)")
    
    # 注入 candidate_id
    ids = list(test_resumes.keys())
    for i, p in enumerate(profiles):
        p["candidate_id"] = ids[i]
    
    # Step 3: 匹配评分
    print("  [3/4] 匹配评分...")
    start = time.time()
    rubric = jd_profile.pop("rubric", None)
    matches = await batch_match_candidates(jd_profile, profiles, rubric=rubric)
    t3 = time.time() - start
    timeline.append(("匹配评分", t3, "ok"))
    print(f"        耗时: {t3:.1f}秒 ({len(matches)}人)")
    
    # Step 4: 排序
    print("  [4/4] 排序...")
    start = time.time()
    ranking = await rank_candidates(matches)
    t4 = time.time() - start
    timeline.append(("排序", t4, "ok"))
    print(f"        耗时: {t4:.1f}秒")
    
    total_time = time.time() - total_start
    
    return {
        "timeline": timeline,
        "total_time": total_time,
        "ranking": ranking,
        "jd_profile": jd_profile,
    }

pipeline_result = asyncio.run(run_full_pipeline_test())
print()
print(f"  总耗时: {pipeline_result['total_time']:.1f}秒")

---
## 四、评估报告

In [ ]:
# ================================================================
# Cell 8: 性能与质量报告
# ================================================================
print("=" * 60)
print("  HireFlow 系统评估报告")
print("=" * 60)
print(f"  报告日期: {report_date}")
print(f"  报告时间: {report_time}")
print()

# --- 性能分析 ---
print("【性能分析】")
timeline = pipeline_result.get("timeline", [])
total = pipeline_result.get("total_time", 0)

for i, (name, t, status) in enumerate(timeline):
    bar = "█" * int(t / max(total, 1) * 30)
    pct = (t / total * 100) if total > 0 else 0
    print(f"  {i+1}. {name:12s}  {t:5.1f}秒  {bar}  ({pct:.0f}%)")

print(f"  {'─' * 40}")
print(f"  总耗时: {total:.1f}秒")
print(f"  候选人处理速度: {3/total:.1f} 人/秒" if total > 0 else "")
print()

# --- 排序结果 ---
print("【排序结果】")
ranking = pipeline_result.get("ranking", {})
for i, c in enumerate(ranking.get("ranked_candidates", [])):
    score = c.get("total_score", 0)
    rec = c.get("recommendation", "")
    cid = c.get("candidate_id", "?")
    print(f"  {i+1}. {cid:6s}  {score:5.1f}分  {rec}")

print()

# --- 问题检测 ---
print("【问题检测】")
issues = []

# 检查性能
if total > 120:
    issues.append(f"⚠️  Pipeline 总耗时 {total:.0f}秒 > 120秒，考虑优化 LLM 调用")

# 检查分数分布
scores = [c.get("total_score", 0) for c in ranking.get("ranked_candidates", [])]
if scores:
    avg_score = sum(scores) / len(scores)
    if max(scores) - min(scores) < 5:
        issues.append(f"⚠️  候选人分数差距过小 ({max(scores)-min(scores):.0f}分)，评分区分度不足")
    if avg_score < 50:
        issues.append(f"⚠️  平均分数偏低 ({avg_score:.0f}分)，可能 JD 要求过高或简历质量差")

# 检查 LLM 模式
if settings.llm.mode == "local":
    issues.append("ℹ️  当前使用本地模型 (速度较慢但免费)")
else:
    issues.append("ℹ️  当前使用云端模型 (速度快但需要API费用)")

if not issues:
    print("  ✅ 未发现明显问题")
else:
    for issue in issues:
        print(f"  {issue}")

print()

# --- 统计摘要 ---
print("【统计摘要】")
summary = ranking.get("summary", {})
total_candidates = summary.get("total_candidates", len(scores))
print(f"  总候选人: {total_candidates}")
print(f"  Strong Match:  {summary.get('strong_match', 0)}")
print(f"  Medium Match:  {summary.get('medium_match', 0)}")
print(f"  Weak Match:    {summary.get('weak_match', 0)}")
print(f"  Not Rec:       {summary.get('not_recommended', 0)}")
if scores:
    print(f"  最高分: {max(scores):.0f}")
    print(f"  最低分: {min(scores):.0f}")
    print(f"  平均分: {sum(scores)/len(scores):.0f}")

print()
print("=" * 60)
print("  报告生成完毕")
print(f"  时间: {report_time}")
print("=" * 60)

---
## 五、Agent 评分细节检查

In [ ]:
# ================================================================
# Cell 9: 显示每个候选人的详细评分
# ================================================================
print("【候选人详细评分】")
print(f"  查看时间: {report_time}")
print()

ranked = ranking.get("ranked_candidates", [])

for i, candidate in enumerate(ranked):
    cid = candidate.get("candidate_id", "?")
    score = candidate.get("total_score", 0)
    rec = candidate.get("recommendation", "?")
    
    print(f"  ┌─ 第{i+1}名: {cid} ({score:.0f}分, {rec})")
    print(f"  │")
    
    # 维度分数
    dims = candidate.get("dimension_scores", {})
    if isinstance(dims, dict):
        for dim_name, dim_score in dims.items():
            if dim_name == "risk_penalty" and dim_score != 0:
                print(f"  │  🔴 {dim_name}: {dim_score}")
            else:
                # 简单的可视化条形图
                max_for_dim = 30 if "technical" in dim_name else (20 if "project" in dim_name else 15)
                bar_len = int(abs(dim_score) / max(max_for_dim, 1) * 15)
                bar = "█" * bar_len + "░" * (15 - bar_len)
                if dim_name != "risk_penalty":
                    print(f"  │  {bar} {dim_name}: {dim_score}")
    
    # 优势 (前2条)
    strengths = candidate.get("strengths", [])
    if strengths:
        print(f"  │")
        print(f"  │  优势:")
        for s in strengths[:2]:
            s_text = str(s)[:80]
            print(f"  │    ✅ {s_text}")
    
    # 风险 (前2条)
    risks = candidate.get("risks", [])
    if risks:
        print(f"  │")
        print(f"  │  风险:")
        for r in risks[:2]:
            r_text = str(r)[:80]
            print(f"  │    ⚠️  {r_text}")
    
    print(f"  └{'─' * 50}")
    print()

---
## 六、文件结构概览

In [ ]:
# ================================================================
# Cell 10: 项目文件结构可视化
# ================================================================
import os

print("【项目文件结构】")
print(f"  扫描时间: {report_time}")
print()

project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "evaluation" else os.getcwd()

# 统计各目录的文件数
stats = {}
total_py_files = 0
total_lines = 0

for root, dirs, files in os.walk(project_root):
    # 跳过不需要的目录
    dirs[:] = [d for d in dirs if d not in ['.git', '.codegraph', '__pycache__', '.claude', 'node_modules']]
    
    py_files = [f for f in files if f.endswith('.py')]
    total_py_files += len(py_files)
    
    for f in py_files:
        try:
            with open(os.path.join(root, f), 'r') as fp:
                total_lines += len(fp.readlines())
        except:
            pass
    
    # 只记录顶层和 app/ 子目录
    rel = os.path.relpath(root, project_root)
    if rel.startswith('app') or rel == '.':
        py_count = len(py_files)
        all_count = len(files)
        if py_count > 0 or rel == '.':
            stats[rel] = {'py_files': py_count, 'all_files': all_count}

# 打印结构
for path, info in sorted(stats.items()):
    indent = "  " * (path.count('/') + 1)
    name = path if path != '.' else 'HireFlowAgents/'
    print(f"{indent}📁 {name}  ({info['py_files']} py, {info['all_files']} files)")

print()
print(f"  总计: {total_py_files} 个 Python 文件, {total_lines} 行代码")
print(f"  模块: app/ (agents/api/graph/schemas/services/database/utils)")

---
## 使用说明

运行本 Notebook 后，会在终端输出完整的评估报告。

### 前置条件
```bash
conda activate hireflowagents
docker compose up -d postgres qdrant
```

### 运行方式
```bash
cd evaluation
jupyter notebook 系统评估报告.ipynb
# 或
jupyter nbconvert --to notebook --execute 系统评估报告.ipynb
```